# Notebook 1 — Ensemble Foundations & Voting Classifiers

**Series:** Ensemble Learning · Notebook 1 of 3  
**Theory depth:** [ensemble_foundations.md](ensemble_foundations.md)  
**What you build here:** Voting classifiers from scratch + sklearn, bias-variance demo, soft/hard comparison

---

## Section 0 — Setup & Version Check

In [ ]:
# ── Imports and version verification ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.special import comb

# Verify sklearn >= 1.4 for full compatibility
assert sklearn.__version__ >= '1.4', f'Need sklearn >= 1.4, got {sklearn.__version__}'
print(f'sklearn version: {sklearn.__version__}')
np.random.seed(42)

## Section 1 — Quick Theory Recap

Single models suffer from either high **bias** (underfitting) or high **variance** (overfitting).  
Ensembles combine multiple models whose errors are uncorrelated — averaging cancels noise, reducing variance.  
Two aggregation strategies: **hard voting** (majority label) and **soft voting** (average probabilities).  
→ Full theory in [ensemble_foundations.md](ensemble_foundations.md)

## Section 2 — Data Preparation

In [ ]:
# ── Toy dataset: nonlinear classification for clear ensemble visualization ───
X_toy, y_toy = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,   # 10 of 20 features actually predict the target
    n_redundant=4,      # 4 features are linear combos of informative ones
    n_clusters_per_class=2,
    random_state=42
)
X_toy_train, X_toy_test, y_toy_train, y_toy_test = train_test_split(
    X_toy, y_toy, test_size=0.2, stratify=y_toy, random_state=42
)
print(f'Toy dataset — Train: {X_toy_train.shape}, Test: {X_toy_test.shape}')

In [ ]:
# ── Real dataset: Wisconsin Breast Cancer (sklearn built-in) ─────────────────
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.2, stratify=y_cancer, random_state=42
)
print(f'Breast Cancer — Train: {X_tr.shape}, Test: {X_te.shape}')
print(f'Class balance: {np.bincount(y_tr)} (train) | {np.bincount(y_te)} (test)')

## Section 3 — Model Implementation

### 3a. The Diversity Argument — From Scratch

Before writing a single sklearn line, let's verify the core math: 5 independent 70%-accurate models give an ~83.7%-accurate ensemble.

In [ ]:
# ── Verify the binomial ensemble accuracy formula ────────────────────────────
def ensemble_accuracy(p_base, n_models):
    """Probability that majority (>n/2) of n independent models are correct."""
    majority_threshold = n_models // 2 + 1
    acc = sum(
        comb(n_models, k, exact=True) * (p_base**k) * ((1 - p_base)**(n_models - k))
        for k in range(majority_threshold, n_models + 1)
    )
    return acc

# Show gains as ensemble size grows (each model: 70% accurate)
sizes = [1, 3, 5, 11, 21, 51, 101]
for n in sizes:
    print(f'{n:>4} models → ensemble accuracy: {ensemble_accuracy(0.70, n):.4f}')

In [ ]:
# ── Hard voting from scratch: majority vote over prediction arrays ───────────
class HardVotingEnsemble:
    def __init__(self, estimators):
        self.estimators = estimators   # list of (name, fitted_model) tuples

    def predict(self, X):
        # Stack each model's predictions as columns
        votes = np.column_stack([est.predict(X) for _, est in self.estimators])
        # Majority vote per row: scipy mode equivalent
        return np.apply_along_axis(
            lambda row: np.bincount(row).argmax(), axis=1, arr=votes
        )

# Build and fit three diverse classifiers on the toy data
lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=300, random_state=42))])
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
svm = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', random_state=42))])

for model in [lr, dt, svm]:
    model.fit(X_toy_train, y_toy_train)

hard_ensemble = HardVotingEnsemble([('lr', lr), ('dt', dt), ('svm', svm)])
hard_pred = hard_ensemble.predict(X_toy_test)
print(f'Hand-coded Hard Voting accuracy: {accuracy_score(y_toy_test, hard_pred):.4f}')

In [ ]:
# ── sklearn VotingClassifier: hard and soft voting ───────────────────────────
# Soft voting needs probability outputs — calibrate SVC (doesn't output proba natively)
lr_pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=300, random_state=42))])
dt_clf  = DecisionTreeClassifier(max_depth=5, random_state=42)
# CalibratedClassifierCV wraps SVC to produce calibrated probabilities
svm_calib = Pipeline([
    ('sc', StandardScaler()),
    ('clf', CalibratedClassifierCV(SVC(kernel='rbf', random_state=42), cv=5, method='isotonic'))
])

hard_voter = VotingClassifier(
    estimators=[('lr', lr_pipe), ('dt', dt_clf), ('svm', svm_calib)],
    voting='hard'
)
soft_voter = VotingClassifier(
    estimators=[('lr', lr_pipe), ('dt', dt_clf), ('svm', svm_calib)],
    voting='soft'
)

results = {}
for name, model in [('LR only', lr_pipe), ('DT only', dt_clf), ('SVM only', svm_calib),
                    ('Hard Voting', hard_voter), ('Soft Voting', soft_voter)]:
    model.fit(X_toy_train, y_toy_train)
    acc = accuracy_score(y_toy_test, model.predict(X_toy_test))
    results[name] = acc
    print(f'{name:>15}: {acc:.4f}')

## Section 4 — Bias-Variance Decomposition Demo

Let's visualize how model complexity drives bias and variance using repeated train/test splits.

In [ ]:
# ── Empirical bias-variance tradeoff: vary tree depth, measure variance ──────
depths = range(1, 16)
N_TRIALS = 30  # number of random train/test splits to estimate variance

mean_accs, std_accs = [], []
for depth in depths:
    trial_accs = []
    for trial in range(N_TRIALS):
        # Different random split each trial — variance comes from this instability
        Xtr, Xte, ytr, yte = train_test_split(X_toy, y_toy, test_size=0.2, random_state=trial)
        dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
        dt.fit(Xtr, ytr)
        trial_accs.append(accuracy_score(yte, dt.predict(Xte)))
    mean_accs.append(np.mean(trial_accs))
    std_accs.append(np.std(trial_accs))

mean_accs, std_accs = np.array(mean_accs), np.array(std_accs)
print('Depth | Mean Acc | Std Dev (variance proxy)')
for d, m, s in zip(depths, mean_accs, std_accs):
    print(f'  {d:>2}  |  {m:.4f}  |  {s:.4f}')

## Section 5 — Visualizations

In [ ]:
# ── Plot bias-variance tradeoff and ensemble comparison ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bias-variance tradeoff curve
ax = axes[0]
depths_list = list(depths)
ax.plot(depths_list, mean_accs, 'o-', color='steelblue', label='Mean Accuracy')
ax.fill_between(depths_list,
                mean_accs - std_accs,
                mean_accs + std_accs,
                alpha=0.3, color='steelblue', label='±1 Std Dev (variance)')
ax.set_xlabel('Tree Depth')
ax.set_ylabel('Test Accuracy')
ax.set_title('Bias-Variance vs. Tree Depth\n(shallow=high bias, deep=high variance)')
ax.legend()
ax.grid(alpha=0.3)

# Right: voting ensemble comparison bar chart
ax2 = axes[1]
colors = ['#78909C', '#78909C', '#78909C', '#4CAF50', '#2196F3']
bars = ax2.bar(list(results.keys()), list(results.values()), color=colors, edgecolor='black')
ax2.set_ylim(0.85, 1.0)
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Individual Models vs Voting Ensembles')
ax2.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, results.values()):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('voting_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Compare soft vs hard voting on breast cancer dataset ─────────────────────
cancer_hard = VotingClassifier(
    estimators=[
        ('lr', Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=300, random_state=42))])),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('svm', Pipeline([
            ('sc', StandardScaler()),
            ('clf', CalibratedClassifierCV(SVC(random_state=42), cv=5))
        ]))
    ],
    voting='hard'
)
cancer_soft = VotingClassifier(
    estimators=[
        ('lr', Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=300, random_state=42))])),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('svm', Pipeline([
            ('sc', StandardScaler()),
            ('clf', CalibratedClassifierCV(SVC(random_state=42), cv=5))
        ]))
    ],
    voting='soft'
)

for name, model in [('Hard Voting', cancer_hard), ('Soft Voting', cancer_soft)]:
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    print(f'{name} on Breast Cancer: {acc:.4f}')
    print(classification_report(y_te, model.predict(X_te), target_names=cancer.target_names))

## Section 6 — Comparison Table

In [ ]:
# ── Styled comparison DataFrame: individual models vs ensembles ──────────────
# Run all models on breast cancer for a fair comparison table
models_to_compare = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=300, random_state=42))]),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM (calibrated)': Pipeline([('sc', StandardScaler()), ('clf', CalibratedClassifierCV(SVC(random_state=42), cv=5))]),
    'Hard Voting': cancer_hard,  # already fitted above
    'Soft Voting': cancer_soft,
    'Random Forest (100 trees)': RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1)
}

rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models_to_compare.items():
    model.fit(X_tr, y_tr)  # re-fit for models not already fitted
    test_acc = accuracy_score(y_te, model.predict(X_te))
    cv_scores = cross_val_score(model, X_cancer, y_cancer, cv=cv, scoring='accuracy')
    rows.append({'Model': name, 'Test Acc': round(test_acc, 4),
                 'CV Mean': round(cv_scores.mean(), 4), 'CV Std': round(cv_scores.std(), 4)})

comparison_df = pd.DataFrame(rows).sort_values('CV Mean', ascending=False).reset_index(drop=True)

# Highlight best values
comparison_df.style.highlight_max(subset=['Test Acc', 'CV Mean'], color='lightgreen') \
                   .highlight_min(subset=['CV Std'], color='lightyellow') \
                   .format({'Test Acc': '{:.4f}', 'CV Mean': '{:.4f}', 'CV Std': '{:.4f}'})

## Section 7 — Key Takeaways

- **Diversity is the engine:** five 70%-accurate independent models produce ~83.7% ensemble accuracy — verify this with the binomial formula, not intuition alone.
- **Soft voting > hard voting when models are calibrated.** Use `CalibratedClassifierCV` for SVMs and decision trees before including them in soft-voting ensembles.
- **Bias-variance tradeoff is observable:** increasing tree depth increases variance (wider confidence band) even when mean accuracy improves.
- **Ensembles don't escape noise:** irreducible error is a floor no model can breach — focus your effort on bias/variance, not magic.
- **Start simple:** a well-configured soft voting ensemble of three diverse models often beats complex single models and is a strong baseline before moving to boosting.